# Линза single-mask для LLaDA-8B-Instruct

Вспомогательная J-линза для конфигурации «реальный контекст + один `[MASK]` в конце», в которой линза читается при оценке и в экспериментах. Корпус строится без сэмплера: каждый текст WikiText обрезается на всех длинах контекста от 16 до 127 токенов и дополняется одним `[MASK]`; из полученных примеров случайно берётся 10 000. Цель — позиция маски, источники — контекст после первых 16 позиций. Бинов нет, линза обучается одним вызовом `fit`.

Выход: `llada_single_mask_lens_1000.pt`.


In [1]:
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM

import jlens
jlens.configure_logging()
from jlens import Layout
from jlens.lens import JacobianLens
from jlens.examples import load_wikitext_prompts

from jlens.diffusion_corpus import build_single_mask_examples, get_mask_token_id, MASK_ID

import os

In [2]:
seed = 42
import random
import numpy as np
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
device = "cuda:0"

## Загрузка LLaDA-8B-Instruct


In [4]:
model_name = "GSAI-ML/LLaDA-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
hf_model = AutoModelForCausalLM.from_pretrained(
    model_name, trust_remote_code=True, dtype=torch.bfloat16,
).to(device)

class WrappedLLaDA:
    def __init__(self, model):
        self.model = model
        self.blocks = model.transformer.blocks
        self.ln_f = model.transformer.ln_f
        self.wte = model.transformer.wte

    def forward(self, input_ids, **kwargs):
        return self.model(input_ids, **kwargs)

    def __call__(self, input_ids, **kwargs):
        return self.forward(input_ids, **kwargs)

hf_model.text_module = WrappedLLaDA(hf_model.model)
hf_model.lm_head = hf_model.model.transformer.ff_out

layout = Layout(path="text_module", layers="blocks", norm="ln_f", embed="wte", lm_head="lm_head")
model = jlens.from_hf(hf_model, tokenizer, layout=layout)

mask_id = get_mask_token_id(hf_model, fallback=MASK_ID)
print(f"mask_id = {mask_id}")
model

Loading checkpoint shards: 100%|██████████| 6/6 [00:00<00:00, 225.33it/s]


mask_id = 126336


HFLensModel(LLaDAModelLM, n_layers=32, d_model=4096)

## Сбор корпуса

`build_single_mask_examples` только токенизирует тексты и дописывает маску, поэтому корпус собирается без обращения к модели.


In [5]:
prompt_texts = load_wikitext_prompts(n_prompts=1000)

examples, source_masks, target_masks = build_single_mask_examples(
    tokenizer, prompt_texts,
    total_len=128,
    prompt_lens=range(16, 128),
    skip_first_source=16,
    mask_id=mask_id,
)
print(f"Собрано {len(examples)} примеров из {len(prompt_texts)} промптов x "
      f"{len(range(16, 128))} вариантов prompt_len (без пропущенных -- см. ниже)")

Собрано 108780 примеров из 1000 промптов x 112 вариантов prompt_len (без пропущенных -- см. ниже)


## Подвыборка

Корпус равномерно покрывает длины контекста по построению, поэтому достаточно простой случайной подвыборки до 10 000 примеров — того же объёма, что и у диффузионной линзы.


In [6]:
TARGET_TOTAL = 10000

rng = random.Random(seed)
if len(examples) > TARGET_TOTAL:
    sampled_idx = rng.sample(range(len(examples)), TARGET_TOTAL)
else:
    sampled_idx = list(range(len(examples)))

examples_sub = [examples[i].to(device) for i in sampled_idx]
source_masks_sub = [source_masks[i] for i in sampled_idx]
target_masks_sub = [target_masks[i] for i in sampled_idx]

print(f"Собрано всего: {len(examples)}")
print(f"После субсэмплирования: {len(examples_sub)}")

Собрано всего: 108780
После субсэмплирования: 10000


## Обучение


In [ ]:
single_mask_lens = jlens.fitting.fit(
    model, examples_sub,
    source_position_masks=source_masks_sub,
    target_position_masks=target_masks_sub,
    dim_batch=32,
    checkpoint_path="./llada_single_mask_ckpt_1000.pt",
    checkpoint_every=50,
)
single_mask_lens.save("./llada_single_mask_lens_1000.pt")
single_mask_lens

[    45s + 45.82s] fit: n_layers=32 d_model=4096, fitting 31 source layers (target=L31) on 10000 examples
[    56s + 11.07s]   example 1/10000  seq_len=23 n_valid=6  11s  max||J||/sqrt(d)=9.217  max_d_mean=nan
[  1m22s + 25.58s]   example 2/10000  seq_len=69 n_valid=52  26s  max||J||/sqrt(d)=0.342  max_d_mean=5.03e-01
[  1m50s + 27.90s]   example 3/10000  seq_len=77 n_valid=60  28s  max||J||/sqrt(d)=0.037  max_d_mean=3.33e-01
[  2m20s + 30.32s]   example 4/10000  seq_len=89 n_valid=72  30s  max||J||/sqrt(d)=0.097  max_d_mean=2.53e-01
[  2m57s + 36.78s]   example 5/10000  seq_len=102 n_valid=85  37s  max||J||/sqrt(d)=0.058  max_d_mean=2.02e-01
[  3m16s + 19.00s]   example 6/10000  seq_len=37 n_valid=20  19s  max||J||/sqrt(d)=0.417  max_d_mean=3.27e-01
[  3m45s + 28.89s]   example 7/10000  seq_len=81 n_valid=64  29s  max||J||/sqrt(d)=0.613  max_d_mean=1.72e-01
[  4m20s + 35.00s]   example 8/10000  seq_len=103 n_valid=86  35s  max||J||/sqrt(d)=0.049  max_d_mean=1.36e-01
[  4m52s + 31.73s]